# Supernan AI Video Dubbing Pipeline

**Steps:** FFmpeg → Whisper → IndicTrans2 → XTTS v2 → Wav2Lip → GFPGAN

> **Required:** Runtime → Change runtime type → **T4 GPU**

Run each cell in order top to bottom.

## 0. Install Dependencies

In [ ]:
# Check GPU
!nvidia-smi

In [ ]:
# Install core Python dependencies
!pip install -q openai-whisper
!pip install -q TTS
!pip install -q googletrans==4.0.0rc1
!pip install -q librosa soundfile pydub
!pip install -q ffmpeg-python
!pip install -q basicsr facexlib realesrgan

In [ ]:
# Clone Wav2Lip
import os
if not os.path.isdir('Wav2Lip'):
    !git clone -q https://github.com/Rudrabha/Wav2Lip.git
    !pip install -q -r Wav2Lip/requirements.txt

# Download Wav2Lip checkpoint (wav2lip — no GAN, faster on free Colab)
os.makedirs('Wav2Lip/checkpoints', exist_ok=True)
if not os.path.exists('Wav2Lip/checkpoints/wav2lip.pth'):
    !wget -q --show-progress \
        'https://iiitaphyd-my.sharepoint.com/personal/radrabha_m_research_iiit_ac_in/_layouts/15/download.aspx?share=EdjI7bZlgApMqsVoEUUXpLsBxqXbn5z5DVux1K-153ZHjg' \
        -O Wav2Lip/checkpoints/wav2lip.pth
    print('wav2lip.pth downloaded')
else:
    print('wav2lip.pth already present')

In [ ]:
# Clone and install GFPGAN
if not os.path.isdir('GFPGAN'):
    !git clone -q https://github.com/TencentARC/GFPGAN.git
    %cd GFPGAN
    !pip install -q -r requirements.txt
    !python setup.py develop -q
    %cd ..

os.makedirs('GFPGAN/experiments/pretrained_models', exist_ok=True)
if not os.path.exists('GFPGAN/experiments/pretrained_models/GFPGANv1.4.pth'):
    !wget -q --show-progress \
        https://github.com/TencentARC/GFPGAN/releases/download/v1.3.4/GFPGANv1.4.pth \
        -O GFPGAN/experiments/pretrained_models/GFPGANv1.4.pth
    print('GFPGANv1.4.pth downloaded')
else:
    print('GFPGANv1.4.pth already present')

In [ ]:
# Clone this repo into Colab
if not os.path.isdir('ai-video-dubbing'):
    !git clone -q https://github.com/sneha31-debug/ai-video-dubbing.git
%cd ai-video-dubbing

## 1. Upload & Extract Clip

Upload your source video using the cell below, then run the extraction.

In [ ]:
from google.colab import files
import os

uploaded = files.upload()  # Upload source_video.mp4
source_video = list(uploaded.keys())[0]
os.makedirs('input', exist_ok=True)
os.rename(source_video, f'input/{source_video}')
source_video = f'input/{source_video}'
print(f'Video uploaded: {source_video}')

In [ ]:
# Configuration — edit these values if needed
START_TIME = 15     # seconds from start
DURATION   = 15     # clip length in seconds

CLIP_PATH  = 'output/clips/clip.mp4'
WAV_PATH   = 'output/audio/clip_audio.wav'

os.makedirs('output/clips', exist_ok=True)
os.makedirs('output/audio', exist_ok=True)

# Extract video clip (stream copy — no re-encode)
!ffmpeg -y -i {source_video} -ss {START_TIME} -t {DURATION} -c copy {CLIP_PATH}

# Extract audio as 16kHz mono WAV (required by Whisper)
!ffmpeg -y -i {CLIP_PATH} -vn -acodec pcm_s16le -ar 16000 -ac 1 {WAV_PATH}

print(f'Clip : {CLIP_PATH}')
print(f'Audio: {WAV_PATH}')

## 2. Transcribe with Whisper

In [ ]:
import whisper
import json, os

os.makedirs('output/transcripts', exist_ok=True)

model = whisper.load_model('small')
result = model.transcribe(WAV_PATH, language='en', word_timestamps=True)

english_text = result['text'].strip()
print('Transcribed text:')
print(english_text)

with open('output/transcripts/transcript.json', 'w') as f:
    json.dump(result, f, ensure_ascii=False, indent=2)
with open('output/transcripts/transcript.txt', 'w') as f:
    f.write(english_text)

## 3. Translate to Hindi

Tries IndicTrans2 first. Falls back to googletrans if VRAM is insufficient.

In [ ]:
os.makedirs('output/translations', exist_ok=True)
hindi_text = None
method = None

# Try IndicTrans2
try:
    !pip install -q git+https://github.com/VarunGumma/IndicTransTokenizer
    from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
    from IndicTransTokenizer import IndicProcessor
    import torch

    model_name = 'ai4bharat/indictrans2-en-indic-1B'
    print(f'Loading IndicTrans2: {model_name}')
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    trans_model = AutoModelForSeq2SeqLM.from_pretrained(model_name, trust_remote_code=True)
    ip = IndicProcessor(inference=True)

    batch = ip.preprocess_batch([english_text], src_lang='eng_Latn', tgt_lang='hin_Deva')
    inputs = tokenizer(batch, truncation=True, padding='longest', return_tensors='pt')
    with torch.no_grad():
        generated = trans_model.generate(**inputs, num_beams=5, max_length=256)
    decoded = tokenizer.batch_decode(generated, skip_special_tokens=True)
    hindi_text = ip.postprocess_batch(decoded, lang='hin_Deva')[0]
    method = 'IndicTrans2'

except Exception as e:
    print(f'IndicTrans2 failed: {e}')
    print('Using googletrans fallback...')
    from googletrans import Translator
    hindi_text = Translator().translate(english_text, dest='hi').text
    method = 'googletrans'

print(f'\nHindi ({method}):')
print(hindi_text)

with open('output/translations/translation.txt', 'w', encoding='utf-8') as f:
    f.write(hindi_text)

## 4. Voice Cloning with XTTS v2

In [ ]:
from TTS.api import TTS
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

tts = TTS('tts_models/multilingual/multi-dataset/xtts_v2').to(device)
raw_hindi_wav = 'output/audio/hindi_audio_raw.wav'

tts.tts_to_file(
    text=hindi_text,
    speaker_wav=WAV_PATH,   # clones the original speaker's voice
    language='hi',
    file_path=raw_hindi_wav
)
print(f'Raw Hindi audio: {raw_hindi_wav}')

In [ ]:
# Time-stretch Hindi audio to exactly match the original clip duration
import librosa
import soundfile as sf

def get_duration(path):
    y, sr = librosa.load(path, sr=None)
    return librosa.get_duration(y=y, sr=sr), y, sr

orig_dur, _, _ = get_duration(WAV_PATH)
hindi_dur, y_hindi, sr_hindi = get_duration(raw_hindi_wav)

print(f'Original clip : {orig_dur:.2f}s')
print(f'Hindi audio   : {hindi_dur:.2f}s')

final_hindi_wav = 'output/audio/hindi_audio.wav'

if abs(hindi_dur - orig_dur) > 0.5:
    stretch_rate = hindi_dur / orig_dur
    print(f'Stretching audio by factor: {stretch_rate:.3f}')
    y_stretched = librosa.effects.time_stretch(y_hindi, rate=stretch_rate)
    sf.write(final_hindi_wav, y_stretched, sr_hindi)
else:
    import shutil
    shutil.copy(raw_hindi_wav, final_hindi_wav)
    print('Durations close — no stretch needed')

print(f'Final Hindi audio: {final_hindi_wav}')

## 5. Lip Sync with Wav2Lip

In [ ]:
LIPSYNCED = 'output/clips/lipsynced.mp4'

!python ../Wav2Lip/inference.py \
    --checkpoint_path ../Wav2Lip/checkpoints/wav2lip.pth \
    --face {CLIP_PATH} \
    --audio {final_hindi_wav} \
    --outfile {LIPSYNCED} \
    --nosmooth

print(f'Lip-synced video: {LIPSYNCED}')

## 6. Face Restoration with GFPGAN

In [ ]:
import shutil

FRAMES_RAW = 'output/clips/frames_raw'
FRAMES_RESTORED = 'output/clips/frames_restored'
os.makedirs(FRAMES_RAW, exist_ok=True)

# Extract frames from lip-synced video
!ffmpeg -y -i {LIPSYNCED} -vf fps=25 {FRAMES_RAW}/frame_%04d.png

# Run GFPGAN on frames
!python ../GFPGAN/inference_gfpgan.py \
    -i {FRAMES_RAW} \
    -o {FRAMES_RESTORED} \
    -v 1.4 \
    --upscale 2 \
    --only_center_face

In [ ]:
# Reassemble final video: restored frames + Hindi audio
FINAL_OUTPUT = 'output/final/hindi_dubbed_final.mp4'
os.makedirs('output/final', exist_ok=True)

# Extract audio from lip-synced video
!ffmpeg -y -i {LIPSYNCED} -vn -acodec pcm_s16le output/audio/lipsynced_audio.wav

# Combine restored frames + audio
!ffmpeg -y \
    -framerate 25 \
    -i {FRAMES_RESTORED}/restored_imgs/frame_%04d.png \
    -i output/audio/lipsynced_audio.wav \
    -c:v libx264 -c:a aac -pix_fmt yuv420p \
    {FINAL_OUTPUT}

# Cleanup raw frames
shutil.rmtree(FRAMES_RAW, ignore_errors=True)

print(f'Final output: {FINAL_OUTPUT}')

## 7. Preview & Download

In [ ]:
from IPython.display import Video
Video(FINAL_OUTPUT, embed=True)

In [ ]:
# Download the final dubbed video
from google.colab import files
files.download(FINAL_OUTPUT)